<h1 style="font-family: 'Times New Roman'; text-align: center; color: #1a1a2e;">
⚙️ PSL Data Analytics — Notebook 03: Feature Engineering (2016–2024)
</h1>

---

**Objective:** Create analytically valuable derived features that power richer EDA and the Streamlit dashboard.

**Input:** Cleaned CSVs from `../data/processed/`  
**Output:** Feature-enriched CSVs saved back to `../data/processed/`


## 1. Setup

In [16]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

PROC = '../data/preprocessed/'
print(" Setup complete")


 Setup complete


## 2. Load Cleaned Datasets

In [17]:
most_runs          = pd.read_csv(PROC + 'most_runs_clean.csv')
most_sixes_history = pd.read_csv(PROC + 'most_sixes_history_clean.csv')
most_wickets       = pd.read_csv(PROC + 'most_wickets_clean.csv')
most_catches       = pd.read_csv(PROC + 'most_catches_clean.csv')
most_dismissals    = pd.read_csv(PROC + 'most_dismissals_clean.csv')
highest_score      = pd.read_csv(PROC + 'highest_score_clean.csv')
most_sixes_innings = pd.read_csv(PROC + 'most_sixes_innings_clean.csv')
best_bowling       = pd.read_csv(PROC + 'best_bowling_clean.csv')
highest_totals     = pd.read_csv(PROC + 'highest_totals_clean.csv')
lowest_totals      = pd.read_csv(PROC + 'lowest_totals_clean.csv')
result_summary     = pd.read_csv(PROC + 'result_summary_clean.csv')
match_wins         = pd.read_csv(PROC + 'match_wins_clean.csv')

print(" All datasets loaded")


 All datasets loaded


## 3. Batting Features

### 3.1 Career Batting Features (`most_runs`)

In [18]:
df = most_runs.copy()

# ── Runs per match (efficiency in shorter appearances) ─────────
# Why: shows how quickly a batter contributes, not just career total
df['runs_per_match'] = (df['runs'] / df['matches']).round(2)

# ── Boundary contribution % (how much of runs come from boundaries) ──
# Why: distinguishes power hitters from rotators
df['boundary_runs']   = df['fours'] * 4
df['boundary_pct']    = (df['boundary_runs'] / df['runs'] * 100).round(1)

# ── Innings utilisation (innings played per match) ──────────────
# Why: players who bat more often per match are higher up the order
df['innings_per_match'] = (df['innings'] / df['matches']).round(3)

# ── Not-out rate ────────────────────────────────────────────────
# Why: high not-out rates inflate batting averages
df['not_out_rate'] = (df['not_outs'] / df['innings'] * 100).round(1)

# ── Duck rate ───────────────────────────────────────────────────
# Why: consistency metric — how often does a player get out for 0?
df['duck_rate'] = (df['ducks'] / df['innings'] * 100).round(1)

# ── Conversion rate (50→100) ────────────────────────────────────
# Why: separates players who convert good starts into centuries
df['conversion_rate'] = (
    df['centuries'] / (df['centuries'] + df['fifties'])
    .replace(0, np.nan) * 100
).round(1)

# ── Career stage label ──────────────────────────────────────────
# Why: contextualises longevity and experience
def career_stage(yrs):
    if yrs == 0: return 'Single Season'
    elif yrs <= 2: return 'Early Career'
    elif yrs <= 5: return 'Developing'
    else: return 'Veteran'

df['career_stage'] = df['career_yrs'].apply(career_stage)

# ── Franchise loyalty ───────────────────────────────────────────
# Why: shows versatility vs. loyalty
df['num_teams'] = df['teams_played'].apply(
    lambda x: len(str(x).split('/')) if pd.notna(x) else 1
)
df['franchise_loyalty'] = df['num_teams'].apply(
    lambda n: 'Loyal' if n == 1 else ('Versatile' if n <= 3 else 'Journeyman')
)

print("Batting features added:", ['runs_per_match','boundary_runs','boundary_pct',
      'innings_per_match','not_out_rate','duck_rate','conversion_rate',
      'career_stage','num_teams','franchise_loyalty'])
df[['player_name','runs_per_match','boundary_pct','not_out_rate',
    'duck_rate','conversion_rate','career_stage','franchise_loyalty']].head(8)


Batting features added: ['runs_per_match', 'boundary_runs', 'boundary_pct', 'innings_per_match', 'not_out_rate', 'duck_rate', 'conversion_rate', 'career_stage', 'num_teams', 'franchise_loyalty']


,player_name,runs_per_match,boundary_pct,not_out_rate,duck_rate,conversion_rate,career_stage,franchise_loyalty
0,Babar Azam,38.93,44.1,12.5,9.1,5.7,Veteran,Versatile
1,Fakhar Zaman,30.06,37.5,1.2,6.0,9.5,Veteran,Loyal
2,Mohammad Rizwan,28.95,36.0,19.4,6.9,4.8,Veteran,Versatile
3,Shoaib Malik,25.67,28.9,19.5,4.6,0.0,Veteran,Versatile
4,RR Rossouw,23.99,38.1,21.2,5.0,16.7,Veteran,Versatile
5,Kamran Akmal,26.29,43.2,2.7,10.8,20.0,Veteran,Loyal
6,Mohammad Hafeez,22.19,36.7,14.9,6.8,0.0,Veteran,Versatile
7,Sarfaraz Ahmed,17.73,35.7,28.8,2.7,0.0,Veteran,Loyal


### 3.2 Batting Efficiency Score (Composite KPI)

In [19]:
# ── Composite batting efficiency score ─────────────────────────

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9) * 100

df['norm_runs']  = minmax(df['runs'])
df['norm_avg']   = minmax(df['average'])
df['norm_sr']    = minmax(df['strike_rate'])

df['bat_efficiency_score'] = (
    0.35 * df['norm_runs'] +
    0.40 * df['norm_avg'] +
    0.25 * df['norm_sr']
).round(2)

# Drop intermediate normalised cols
df = df.drop(columns=['norm_runs','norm_avg','norm_sr'])

print("Top 10 batters by Batting Efficiency Score:")
df.nlargest(10, 'bat_efficiency_score')[
    ['player_name','runs','average','strike_rate','bat_efficiency_score']
]


Top 10 batters by Batting Efficiency Score:


,player_name,runs,average,strike_rate,bat_efficiency_score
0,Babar Azam,3504,45.50,127.41,78.26
34,Usman Khan,747,53.35,159.27,65.29
2,Mohammad Rizwan,2403,41.43,127.48,63.44
88,HC Brook,264,52.80,171.42,63.10
45,TH David,605,43.21,179.52,60.43
1,Fakhar Zaman,2525,30.42,140.27,58.54
17,KA Pollard,1149,38.30,162.28,57.14
3,Shoaib Malik,2336,33.37,128.07,55.90
4,RR Rossouw,2015,31.98,142.70,55.31
24,L Ronchi,1020,36.42,166.12,55.21


In [20]:
most_runs_feat = df.copy()
print(" most_runs feature engineering done | shape:", df.shape)


 most_runs feature engineering done | shape: (150, 29)


## 4. Bowling Features

### 4.1 Career Bowling Features (`most_wickets`)

In [21]:
df = most_wickets.copy()

# ── Wickets per match ──────────────────────────────────────────
# Why: normalises productivity across different career lengths
df['wickets_per_match'] = (df['wickets'] / df['matches']).round(3)

# ── Wickets per over ──────────────────────────────────────────
# Why: raw attack rate, different from economy which is runs
overs_float = df['overs'].astype(str).apply(
    lambda o: int(o.split('.')[0]) + int(o.split('.')[1])/6 if '.' in str(o) else float(o)
)
df['wickets_per_over'] = (df['wickets'] / overs_float).round(3)

# ── Dot ball % proxy ──────────────────────────────────────────
# Why: bowlers who get wickets cheaply are most valuable in T20
# We can proxy this: (1 - economy/12) gives tightness score
df['tightness_score'] = ((1 - df['economy'] / 12) * 100).clip(0, 100).round(1)

# ── Bowling impact score (composite) ──────────────────────────
# Why: single KPI combining wicket-taking and economy
# Weights: Wickets 40% | Bowling SR 35% | Economy 25%
def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-9) * 100
def minmax_inv(s): return 100 - minmax(s)  # lower is better

df['norm_wkts']    = minmax(df['wickets'])
df['norm_sr_inv']  = minmax_inv(df['bowling_sr'])   # lower SR = better
df['norm_econ_inv']= minmax_inv(df['economy'])       # lower econ = better

df['bowl_impact_score'] = (
    0.40 * df['norm_wkts'] +
    0.35 * df['norm_sr_inv'] +
    0.25 * df['norm_econ_inv']
).round(2)

df = df.drop(columns=['norm_wkts','norm_sr_inv','norm_econ_inv'])

# ── Specialist flag ────────────────────────────────────────────
# Why: 5-wicket hauls and 4-wicket hauls are match-winning performances
df['is_match_winner'] = (df['five_wicket_hauls'] > 0) | (df['four_wicket_hauls'] >= 2)

# ── Franchise tag ─────────────────────────────────────────────
df['num_teams'] = df['teams_played'].apply(
    lambda x: len(str(x).split('/')) if pd.notna(x) else 1
)

print("Top 10 bowlers by Bowling Impact Score:")
df.nlargest(10, 'bowl_impact_score')[
    ['player_name','wickets','bowling_avg','economy','bowl_impact_score']
]


Top 10 bowlers by Bowling Impact Score:


,player_name,wickets,bowling_avg,economy,bowl_impact_score
0,Wahab Riaz,113,22.68,7.79,83.27
1,Hasan Ali,108,22.96,8.03,80.44
2,Shaheen Shah Afridi,103,21.02,8.00,80.23
3,Shadab Khan,91,25.38,7.68,72.74
17,Rashid Khan,44,15.47,6.13,68.05
4,Faheem Ashraf,78,24.92,8.42,66.17
10,Imran Tahir,56,19.64,7.05,66.07
5,Mohammad Amir,73,28.68,7.57,63.17
6,Mohammad Nawaz,72,28.65,7.62,62.70
9,Usama Mir,59,21.67,8.21,62.11


In [22]:
most_wickets_feat = df.copy()
print(" most_wickets feature engineering done | shape:", df.shape)


 most_wickets feature engineering done | shape: (100, 25)


## 5. Team Features

### 5.1 Result Summary Features

In [23]:
df = result_summary.copy()

# ── Effective matches (excluding no-results) ───────────────────
df['effective_matches'] = df['matches'] - df['no_result']

# ── Win pct from effective matches ────────────────────────────
df['win_pct_effective'] = (df['won'] / df['effective_matches'] * 100).round(2)

# ── Performance tier ──────────────────────────────────────────
def perf_tier(wp):
    if wp >= 55: return 'Elite'
    elif wp >= 48: return 'Competitive'
    else: return 'Struggling'

df['performance_tier'] = df['win_pct'].apply(perf_tier)

# ── Seasons active ────────────────────────────────────────────
df['seasons_active'] = df['span_end'] - df['span_start'] + 1

print("Team features added:")
df[['team','matches','win_pct','win_pct_effective','performance_tier','seasons_active']]


Team features added:


,team,matches,win_pct,win_pct_effective,performance_tier,seasons_active
0,Islamabad United,100,55.00,55.00,Elite,9
1,Karachi Kings,95,37.89,38.71,Struggling,9
2,Lahore Qalandars,94,42.55,42.55,Struggling,9
3,Multan Sultans,79,56.96,58.44,Elite,7
4,Peshawar Zalmi,104,52.88,53.40,Competitive,9
5,Quetta Gladiators,92,47.82,48.35,Struggling,9


In [24]:
result_summary_feat = df.copy()
print(" result_summary feature engineering done")


 result_summary feature engineering done


## 6. Match-Level Features

### 6.1 Highest Totals

In [25]:
df = highest_totals.copy()

# ── Run rate ─────────────────────────────────────────────────
df['run_rate']        = (df['rr']).round(2)  # already in data as 'rr'

# ── Is powerplay dominant? (RR > 12 suggests explosive opening) ──
df['is_explosive']    = df['run_rate'] >= 12

# ── Innings type ─────────────────────────────────────────────
df['innings_type']    = df['inns'].map({1: 'Batting First', 2: 'Chasing'})

# ── Won while batting explosive ───────────────────────────────
df['explosive_win']   = df['is_explosive'] & df['won']

# ── Year bin ─────────────────────────────────────────────────
df['era'] = pd.cut(df['match_year'], bins=[2015,2018,2021,2025],
                   labels=['Early PSL (2016-18)', 'Mid PSL (2019-21)', 'Modern PSL (2022-24)'])

print("Highest totals features added")
df[['team','runs','run_rate','innings_type','won','is_explosive','era']].head(8)


Highest totals features added


,team,runs,run_rate,innings_type,won,is_explosive,era
0,Multan Sultans,262,13.10,Batting First,True,True,Modern PSL (2022-24)
1,Quetta Gladiators,253,12.65,Chasing,False,True,Modern PSL (2022-24)
2,Islamabad United,247,12.35,Batting First,True,True,Mid PSL (2019-21)
3,Multan Sultans,245,12.25,Batting First,True,True,Modern PSL (2022-24)
4,Multan Sultans,244,12.73,Chasing,True,True,Modern PSL (2022-24)
5,Quetta Gladiators,243,13.25,Chasing,True,True,Modern PSL (2022-24)
6,Peshawar Zalmi,242,12.10,Batting First,False,True,Modern PSL (2022-24)
7,Lahore Qalandars,241,12.05,Batting First,True,True,Modern PSL (2022-24)


In [26]:
highest_totals_feat = df.copy()

# ── Same for lowest totals ─────────────────────────────────────
df2 = lowest_totals.copy()
df2['innings_type'] = df2['inns'].map({1: 'Batting First', 2: 'Chasing'})
df2['era'] = pd.cut(df2['match_year'], bins=[2015,2018,2021,2025],
                    labels=['Early PSL (2016-18)', 'Mid PSL (2019-21)', 'Modern PSL (2022-24)'])
lowest_totals_feat = df2.copy()
print(" highest_totals + lowest_totals features done")


 highest_totals + lowest_totals features done


## 7. Timeline Features

In [27]:
df = match_wins.copy()
team_cols = [c for c in df.columns if c != 'match_number']

# ── Win gap (how many matches between each win, per team) ─────
# We compute the rank at each match — diff gives wins in that batch
for col in team_cols:
    df[f'{col}_match_wins'] = df[col].diff().fillna(df[col]).astype(int)

# ── Leader at each match ──────────────────────────────────────
df['leader'] = df[team_cols].idxmax(axis=1)

# ── Gap between 1st and 2nd ───────────────────────────────────
def rank_gap(row):
    vals = sorted(row[team_cols].values, reverse=True)
    return vals[0] - vals[1]

df['lead_gap'] = df.apply(rank_gap, axis=1)

match_wins_feat = df.copy()
print(" match_wins feature engineering done | shape:", df.shape)
df.tail(5)


 match_wins feature engineering done | shape: (278, 15)


,match_number,Quetta Gladiators,Karachi Kings,Peshawar Zalmi,Islamabad United,Lahore Qalandars,Multan Sultans,Quetta Gladiators_match_wins,Karachi Kings_match_wins,Peshawar Zalmi_match_wins,Islamabad United_match_wins,Lahore Qalandars_match_wins,Multan Sultans_match_wins,leader,lead_gap
273,274,43,37,56,53,41,44,0,0,0,0,0,1,Peshawar Zalmi,3
274,275,43,37,56,53,41,45,0,0,0,0,0,1,Peshawar Zalmi,3
275,276,43,37,56,54,41,45,0,0,0,1,0,0,Peshawar Zalmi,2
276,277,43,37,56,55,41,45,0,0,0,1,0,0,Peshawar Zalmi,1
277,278,43,37,56,56,41,45,0,0,0,1,0,0,Peshawar Zalmi,0


## 8. Save Feature-Enriched Datasets

In [28]:
saves = {
    'most_runs_feat':          most_runs_feat,
    'most_wickets_feat':       most_wickets_feat,
    'result_summary_feat':     result_summary_feat,
    'highest_totals_feat':     highest_totals_feat,
    'lowest_totals_feat':      lowest_totals_feat,
    'match_wins_feat':         match_wins_feat,
}

for fname, df in saves.items():
    df.to_csv(PROC + fname + '.csv', index=False)
    print(f"   {fname}.csv  →  {df.shape}")

print("\n Feature-enriched datasets saved to ../data/preprocessed/")


   most_runs_feat.csv  →  (150, 29)
   most_wickets_feat.csv  →  (100, 25)
   result_summary_feat.csv  →  (6, 20)
   highest_totals_feat.csv  →  (100, 16)
   lowest_totals_feat.csv  →  (200, 13)
   match_wins_feat.csv  →  (278, 15)

 Feature-enriched datasets saved to ../data/preprocessed/


## 9. Export Feature-Enriched Data to Excel

In [29]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

excel_path = PROC + 'psl_feature_engineered.xlsx'

HEADER_FILL = PatternFill('solid', start_color='2E5E3E', end_color='2E5E3E')
HEADER_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=11)
BODY_FONT   = Font(name='Arial', size=10)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for fname, df in saves.items():
        sheet_name = fname.replace('_feat', '')[:31]
        df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = openpyxl.load_workbook(excel_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 3, 35)
        for cell in col_cells[1:]:
            cell.font = BODY_FONT
    ws.freeze_panes = 'A2'

wb.save(excel_path)
print(f" Excel workbook saved: {excel_path}")
print(f"   Sheets: {wb.sheetnames}")


 Excel workbook saved: ../data/preprocessed/psl_feature_engineered.xlsx
   Sheets: ['most_runs', 'most_wickets', 'result_summary', 'highest_totals', 'lowest_totals', 'match_wins']


## 10. Feature Engineering Summary

| Feature | Dataset | Justification |
|---------|---------|---------------|
| `runs_per_match` | Batting | Efficiency metric, penalises players with many low-output innings |
| `boundary_pct` | Batting | Distinguishes power hitters from rotation batters |
| `not_out_rate` | Batting | Contextualises inflated averages for finishers |
| `duck_rate` | Batting | Consistency indicator |
| `conversion_rate` | Batting | 50-to-100 conversion; separates clutch players |
| `career_stage` | Batting/Bowling | Veterans vs. new entrants |
| `franchise_loyalty` | Batting/Bowling | Loyalty signal — useful for team identity analysis |
| `bat_efficiency_score` | Batting | Composite KPI: 35% runs + 40% avg + 25% SR |
| `wickets_per_match` | Bowling | Volume-normalised productivity |
| `wickets_per_over` | Bowling | Attack rate |
| `tightness_score` | Bowling | Proxy dot-ball control derived from economy rate |
| `bowl_impact_score` | Bowling | Composite: 40% wickets + 35% SR + 25% economy |
| `is_match_winner` | Bowling | Flag for 5-wkt hauls or 2+ four-wkt hauls |
| `win_pct_effective` | Team | Excludes no-result matches for true win rate |
| `performance_tier` | Team | Elite / Competitive / Struggling classification |
| `seasons_active` | Team | Tenure in the league |
| `innings_type` | Totals | First innings vs. chase context |
| `is_explosive` | Totals | High run-rate batting flag |
| `era` | Totals | PSL phase — Early/Mid/Modern |
| `leader` | Timeline | Who was ahead after each match |
| `lead_gap` | Timeline | Gap between 1st and 2nd at each match number |
